9 CLASSIFIER MODELS USING OHLCV DATA+ TECHNICAL INDICATORS

In [ ]:
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression, RidgeClassifier, SGDClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import accuracy_score

# ==========================================
# 1. DATA LOADING & ENHANCED PRE-PROCESSING
# ==========================================
df = pd.read_csv('NIFTY50.csv')
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values('Date').set_index('Date')

# Resample to Weekly (Captures the "Smart Money" weekly flow)
df_w = df.resample('W-FRI').agg({
    'Open': 'first', 'High': 'max', 'Low': 'min', 'Close': 'last', 'Adj Close': 'last'
})

# --- FEATURE ENGINEERING (Stationary + Structural) ---
# Stationary Z-Scores
df_w['MACD'] = df_w['Adj Close'].ewm(span=12).mean() - df_w['Adj Close'].ewm(span=26).mean()
df_w['MACD_Z'] = (df_w['MACD'] - df_w['MACD'].rolling(52).mean()) / df_w['MACD'].rolling(52).std()

df_w['Dist_SMA'] = (df_w['Adj Close'] - df_w['Adj Close'].rolling(20).mean()) / df_w['Adj Close'].rolling(20).mean()
df_w['Dist_SMA_Z'] = (df_w['Dist_SMA'] - df_w['Dist_SMA'].rolling(52).mean()) / df_w['Dist_SMA'].rolling(52).std()

# OHLC Structural Features
df_w['Body_Pct'] = (df_w['Close'] - df_w['Open']) / df_w['Open']
df_w['Vol_Shock'] = ((df_w['High'] - df_w['Low']) / df_w['Low']) / ((df_w['High'] - df_w['Low']) / df_w['Low']).rolling(20).mean()
df_w['Upper_Shadow'] = (df_w['High'] - df_w[['Open', 'Close']].max(axis=1)) / df_w['Open']
df_w['Lower_Shadow'] = (df_w[['Open', 'Close']].min(axis=1) - df_w['Low']) / df_w['Open']

# Market Regime (200-SMA equivalent for weekly)
df_w['Trend_Regime'] = np.where(df_w['Adj Close'] > df_w['Adj Close'].rolling(40).mean(), 1, 0)

# --- VOLATILITY-ADJUSTED TARGETING ---
df_w['Next_Week_Ret'] = df_w['Adj Close'].pct_change().shift(-1)
df_w['Vol_Threshold'] = ((df_w['High'] - df_w['Low']) / df_w['Low']).rolling(20).mean() * 0.5
df_w['Target'] = np.where(df_w['Next_Week_Ret'] > df_w['Vol_Threshold'], 1, 0)

df_w.dropna(inplace=True)

# Define Research Features
features = ['MACD_Z', 'Dist_SMA_Z', 'Vol_Shock', 'Body_Pct', 'Upper_Shadow', 'Lower_Shadow', 'Trend_Regime']
X = df_w[features]
y = df_w['Target']

# ==========================================
# 2. MODEL DEFINITIONS (9 MODELS)
# ==========================================
models = {
    "1. Logistic Regression": LogisticRegression(C=0.5, max_iter=1000),
    "2. Polynomial (Deg 2)": LogisticRegression(C=0.5, max_iter=1000),
    "3. Ridge Classifier": RidgeClassifier(),
    "4. Lasso (L1)": SGDClassifier(loss='log_loss', penalty='l1', random_state=42),
    "5. Elastic Net": SGDClassifier(loss='log_loss', penalty='elasticnet', random_state=42),
    "6. Decision Tree": DecisionTreeClassifier(max_depth=4, random_state=42),
    "7. Random Forest": RandomForestClassifier(n_estimators=200, max_depth=5, random_state=42),
    "8. SVM (RBF)": SVC(kernel='rbf', C=0.5, probability=True),
    "9. XGBoost": XGBClassifier(n_estimators=100, learning_rate=0.05, max_depth=3, eval_metric='logloss', random_state=42)
}

# ==========================================
# 3. WALK-FORWARD EVALUATION
# ==========================================
tscv = TimeSeriesSplit(n_splits=5)
scaler = StandardScaler()
results = {name: [] for name in models.keys()}

for train_idx, test_idx in tscv.split(X):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    # Pre-process Scaling
    X_train_sc = scaler.fit_transform(X_train)
    X_test_sc = scaler.transform(X_test)

    for name, model in models.items():
        if "Polynomial" in name:
            poly = PolynomialFeatures(degree=2, interaction_only=True)
            X_p_tr = poly.fit_transform(X_train_sc)
            X_p_te = poly.transform(X_test_sc)
            model.fit(X_p_tr, y_train)
            acc = accuracy_score(y_test, model.predict(X_p_te))

        elif any(tree_mod in name for tree_mod in ["Tree", "Forest", "XGBoost"]):
            # Tree-based models work best on unscaled, raw structural data
            model.fit(X_train, y_train)
            acc = accuracy_score(y_test, model.predict(X_test))

        else:
            # Linear/Distance models require Z-score scaling
            model.fit(X_train_sc, y_train)
            acc = accuracy_score(y_test, model.predict(X_test_sc))

        results[name].append(acc)

# ==========================================
# 4. FINAL RESULTS & RANKING
# ==========================================
summary = pd.DataFrame({
    'Model': results.keys(),
    'Avg Accuracy': [np.mean(v) for v in results.values()],
    'Stability (Std Dev)': [np.std(v) for v in results.values()]
}).sort_values(by='Avg Accuracy', ascending=False)

print("\n--- RESEARCH ACCURACY REPORT ---")
print(summary.to_string(index=False))



/tmp/ipython-input-1394887428.py:42: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  df_w['Next_Week_Ret'] = df_w['Adj Close'].pct_change().shift(-1)



--- RESEARCH ACCURACY REPORT ---
                 Model  Avg Accuracy  Stability (Std Dev)
          8. SVM (RBF)          0.73                 0.03
1. Logistic Regression          0.72                 0.02
   3. Ridge Classifier          0.72                 0.03
      7. Random Forest          0.71                 0.02
 2. Polynomial (Deg 2)          0.71                 0.02
      6. Decision Tree          0.70                 0.04
            9. XGBoost          0.68                 0.02
        5. Elastic Net          0.68                 0.07
         4. Lasso (L1)          0.66                 0.08
